## **1. Policy Gradient – Intelligent Traffic Signal Control**

In [2]:
!pip install torch torchvision torchaudio
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(4,32),nn.ReLU(),nn.Linear(32,2),nn.Softmax(dim=-1))
    def forward(self,x):
        return self.net(x)
policy=Policy()
optimizer=optim.Adam(policy.parameters(),lr=0.001)
for episode in range(100):
    state=torch.randn(4)
    log_probs=[]
    rewards=[]
    for t in range(20):
        probs=policy(state)
        dist=torch.distributions.Categorical(probs)
        action=dist.sample()
        log_probs.append(dist.log_prob(action))
        reward=1.0 if action.item()==0 and state[0]<0 else -1.0
        rewards.append(reward)
        state=torch.randn(4)
    G=sum(rewards)
    loss=-sum(log_probs)*G
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("Traffic Signal Policy Trained")

   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/122.1 MB 2.4 MB/s eta 0:00:51
   ---------------------------------------- 1.0/122.1 MB 1.8 MB/s eta 0:01:08
   ---------------------------------------- 1.3/122.1 MB 1.6 MB/s eta 0:01:18
   ---------------------------------------- 1.3/122.1 MB 1.6 MB/s eta 0:01:18
    --------------------------------------- 1.6/122.1 MB 1.2 MB/s eta 0:01:41
    --------------------------------------- 1.6/122.1 MB 1.2 MB/s eta 0:01:41
    --------------------------------------- 1.6/122.1 MB 1.2 MB/s eta 0:01:41
    --------------------------------------- 1.8/122.1 MB 923.6 kB/s eta 0:02:11
    --------------------------------------- 1.8/122.1 MB 923.6 kB/s eta 0:02:11
    --------------------------------------- 2.1/122.1 MB 896.4 kB/s eta 0:02:14
    --------------------------------------- 2.4/122.1 MB 932.1 kB/s eta 

## **2. Actor-Critic – Autonomous Warehouse Robot**

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Linear(6,64)
        self.actor=nn.Linear(64,4)
        self.critic=nn.Linear(64,1)
    def forward(self,x):
        h=torch.relu(self.fc(x))
        return torch.softmax(self.actor(h),-1),self.critic(h)

model=ActorCritic()
optimizer=optim.Adam(model.parameters(),lr=0.001)
gamma=0.99

for episode in range(100):
    state=torch.randn(6)
    log_probs=[]
    values=[]
    rewards=[]
    for t in range(20):
        probs,value=model(state)
        dist=torch.distributions.Categorical(probs)
        action=dist.sample()
        reward=1 if action.item()==0 else -0.1
        log_probs.append(dist.log_prob(action))
        values.append(value)
        rewards.append(reward)
        state=torch.randn(6)
    G=sum(rewards)
    advantage=G-torch.stack(values).mean()
    loss=-torch.stack(log_probs).sum()*advantage+advantage.pow(2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Warehouse Robot Model Trained")

Warehouse Robot Model Trained


## **3. DDPG – Continuous Portfolio Optimization**

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(5,64),nn.ReLU(),nn.Linear(64,2),nn.Sigmoid())
    def forward(self,x):
        return self.net(x)
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(7,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,s,a):
        return self.net(torch.cat([s,a],1))
actor=Actor()
critic=Critic()
ao=optim.Adam(actor.parameters(),lr=0.001)
co=optim.Adam(critic.parameters(),lr=0.001)
for episode in range(100):
    state=torch.randn(1,5)
    action=actor(state)
    reward=torch.randn(1,1)
    value=critic(state,action)
    loss_c=(value-reward).pow(2).mean()
    co.zero_grad()
    loss_c.backward()
    co.step()
    loss_a=-critic(state,actor(state)).mean()
    ao.zero_grad()
    loss_a.backward()
    ao.step()
print("Portfolio DDPG Model Trained")

Portfolio DDPG Model Trained


## **4. PPO – Smart HVAC Energy Management**

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
class PPO(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(5,64),nn.ReLU())
        self.actor=nn.Linear(64,3)
        self.critic=nn.Linear(64,1)
    def forward(self,x):
        h=self.net(x)
        return torch.softmax(self.actor(h),-1),self.critic(h)
model=PPO()
optimizer=optim.Adam(model.parameters(),lr=0.001)
for episode in range(100):
    state=torch.randn(1,5)
    probs,value=model(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    reward=-(state[0,0].abs()+action.float())
    advantage=reward-value.squeeze()
    loss=-(dist.log_prob(action)*advantage)+advantage.pow(2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("HVAC PPO Model Trained")

HVAC PPO Model Trained


## **5. TRPO – Industrial Robotic Arm**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(6,64),nn.Tanh(),nn.Linear(64,2))
    def forward(self,x):
        return self.net(x)
policy=Policy()
optimizer=optim.Adam(policy.parameters(),lr=0.0005)
for episode in range(100):
    state=torch.randn(1,6)
    action=policy(state)
    reward=-((action-state[:,:2])**2).mean()
    loss=-reward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(),0.5)
    optimizer.step()
print("Robotic Arm TRPO Model Trained")

Robotic Arm TRPO Model Trained


## **6. A2C vs PPO – Healthcare Treatment Planning**

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
class A2C(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Linear(6,64)
        self.actor=nn.Linear(64,3)
        self.critic=nn.Linear(64,1)
    def forward(self,x):
        h=torch.relu(self.fc(x))
        return torch.softmax(self.actor(h),-1),self.critic(h)
model=A2C()
optimizer=optim.Adam(model.parameters(),lr=0.001)
for episode in range(100):
    state=torch.randn(1,6)
    probs,value=model(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    reward=1 if action.item()==0 else -0.2
    advantage=reward-value.squeeze()
    loss=-dist.log_prob(action)*advantage+advantage.pow(2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("Healthcare A2C Model Trained")

Healthcare A2C Model Trained


## **7. PPO – Autonomous Drone Navigation**

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
class DronePolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(8,64),nn.ReLU(),nn.Linear(64,4),nn.Softmax(dim=-1))
    def forward(self,x):
        return self.net(x)
policy=DronePolicy()
optimizer=optim.Adam(policy.parameters(),lr=0.001)
for episode in range(100):
    state=torch.randn(1,8)
    probs=policy(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    reward=1.0 if action.item()==0 else -0.1
    loss=-dist.log_prob(action)*reward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("Drone PPO Model Trained")

Drone PPO Model Trained


## **8. A3C – Dynamic Cloud Resource Allocation**

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

class A3C(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Linear(5,64)
        self.actor=nn.Linear(64,3)
        self.critic=nn.Linear(64,1)
    def forward(self,x):
        h=torch.relu(self.fc(x))
        return torch.softmax(self.actor(h),-1),self.critic(h)

model=A3C()
optimizer=optim.Adam(model.parameters(),lr=0.001)

for episode in range(100):
    state=torch.randn(1,5)
    probs,value=model(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    reward=1.0-state[0,0].abs()
    advantage=reward-value.squeeze()
    loss=-dist.log_prob(action)*advantage+advantage.pow(2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Cloud Resource A3C Model Trained")

Cloud Resource A3C Model Trained


## **9. PPO – Predictive Maintenance**

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

class MaintenancePolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(6,64),nn.ReLU(),nn.Linear(64,3),nn.Softmax(dim=-1))
    def forward(self,x):
        return self.net(x)

policy=MaintenancePolicy()
optimizer=optim.Adam(policy.parameters(),lr=0.001)

for episode in range(100):
    state=torch.randn(1,6)
    probs=policy(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    reward=1.0 if action.item()==1 else -0.2
    loss=-dist.log_prob(action)*reward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Predictive Maintenance PPO Model Trained")

Predictive Maintenance PPO Model Trained


## **10. PPO – Autonomous Lane-Keeping**

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim

class LanePolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(5,64),nn.Tanh(),nn.Linear(64,3),nn.Softmax(dim=-1))
    def forward(self,x):
        return self.net(x)

policy=LanePolicy()
optimizer=optim.Adam(policy.parameters(),lr=0.001)

for episode in range(100):
    state=torch.randn(1,5)
    probs=policy(state)
    dist=torch.distributions.Categorical(probs)
    action=dist.sample()
    deviation=state[0,0].abs()
    reward=-deviation if action.item()!=1 else 1-deviation
    loss=-dist.log_prob(action)*reward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Lane Keeping PPO Model Trained")

Lane Keeping PPO Model Trained
